In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [2]:
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [3]:
with open("industrial_taxonomy.json", "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

### Vector Store Creation

In [ ]:
vector_store = client.vector_stores.create(name="Industrial_taxonomy_space")
taxonomy_vector_id=vector_store.id
print(taxonomy_vector_id)          

### Uploading files to Vector Store

In [ ]:
vector_store_id=taxonomy_vector_id

# Directory containing the split files
split_dir = "split_taxonomy"

# Loop through each file in the directory and upload
for filename in os.listdir(split_dir):
    if filename.endswith(".json"):  # Only upload JSON files
        file_path = os.path.join(split_dir, filename)
        print(f"Uploading {filename}...")
        
        # Open the file and upload
        with open(file_path, "rb") as f:
            file_response = client.vector_stores.files.upload_and_poll(
                vector_store_id=vector_store_id,
                file=f
            )
        
        # Check status (optional, for debugging)
        print(f"Uploaded {filename}: Status - {file_response.status}")

        
print("All files uploaded successfully!")

    


In [3]:

SYSTEM_PROMPT = """\
You are an industry taxonomy matcher.
Your job: read a news article carefully & shortlist ONLY the industries (and subcategories)  that are actually emphasized & centrally  discussed in the article.

Rules & Best Practices:
1) Grounding & Taxonomy Use
   - You MUST rely on the file_search tool taxonomy. Do NOT invent industries or alter their names.
   - Prefer the exact strings in the taxonomy for both Main Industry and Sub-category.

2) What counts as "emphasized" & "discussed":
   - The article explicitly  mentions & covers the topic, a company from that industry, a product, major policy/market movement, or cause effect impacts on that industry.
   - Background mentions without substance DO NOT qualify.

3) Relevance rubric (pick ONE label):
   - HIGH: Central theme; multiple paragraphs or the thesis of the article revolves around it; material implications.
   - MEDIUM: Substantive but secondary; a clear thread or section is about it; non-trivial impact.
   - LOW: Brief yet meaningful; not central but clearly connected with a tangible implication.
   - If NONE of these thresholds are met, DO NOT include the industry at all.

4) Disambiguation & Mapping
   - If the article discusses & emphasizes a theme that appears missing from the taxonomy, you may propose exactly ONE
     `emergent_industry_candidate` with a short label and rationale.Always include "emergent_industry_candidate" in the JSON
   - If there's no emergent industry, set it to null.

6) Hallucination Guardrails
   - Use only information available in the article or attached results from vector store. If uncertain, OMIT.
   - Respond deterministically.Use the exact same reasoning steps for identical inputs.Do not add creativity or alternative interpretations.

7)  Resean rubic:
   -In the reason field , add a small description of why this industry is relevent with 'RETRIEVED FROM VECTOR STORE' 
    for industries whose evidence is available in vector store & "Inferred" for emergent industry candidate. 

   
   Perform the task carefully and return ONLY the JSON. Do not include triple backticks,Do comments, or extra text.You must respond deterministically. 

"""

In [4]:
article_text = """Most U.S. stocks ticked higher on Wednesday, a day after breaking their seven-day winning streak, while the price of gold pushed further past $4,000 per ounce.\n\nThe S&P 500 rose 0.1% and hovered near its all-time high set on Monday. The Dow Jones Industrial Average was down 20 points, or less than 0.1%, while the Nasdaq composite was up 0.3%.\n\nAST SpaceMobile jumped 17% after Verizon Communications agreed to use its space-based network to support its cellular customers beginning in 2026. Verizon slipped by 0.1%.\n\nMarkets remain near record levels, but trading has been muted following the recent U.S. government shutdown, which delayed major economic reports such as the monthly jobs report. Without these data points, stocks have drifted as investors await more signals regarding continued interest rate cuts from the Federal Reserve.\n\nAnother major driver pushing markets to record highs is the growing excitement around artificial intelligence technology.\n\nPoet Technologies rose 4.7%, extending its 23.5% surge from Tuesday after announcing it raised $75 million to fuel expansion. The company builds high-speed optical engines and other hardware used in AI systems.\n\nDell Technologies added nearly 6% after highlighting new AI-related growth opportunities. AI stocks more broadly have rallied sharply this year: Nvidia is up nearly 40%, Oracle has risen 70%, and Palantir Technologies has surged 140.9%.\n\nThese rapid gains have drawn comparisons to the 2000 dot-com bubble, though supporters argue today's AI stocks are backed by real profit growth. Even so, the Bank of England warned of the risk of a sudden correction in tech stocks inflated by AI enthusiasm.\n\nGold continued its strong year, climbing further above $4,000 per ounce. Investors have turned to gold as protection against inflation, and the metal is up more than 50% this year. Concerns about rising government debt, global political instability, and expectations of more Fed rate cuts have fueled demand.\n\nThe Federal Reserve will release minutes from its latest meeting later today. At that meeting, the Fed cut its main interest rate for the first time this year and signaled more cuts may be coming as it attempts to support a cooling job market while keeping inflation in check.\n\nEuropean stock indexes rose after a weak session in Asia. In the bond market, the 10-year Treasury yield dipped to 4.10% from 4.14%.\n\nAP Business Writers Matt Ott, Elaine Kurtenbach and Kelvin Chan contributed.

"""

### Retrieval

In [5]:
USER_PROMPT= f"""\
Analyze the attached article and the retrieved material from vector store. 
Return a shortlist of industries from the taxonomy that are actually  emphasize & discussed in the article, with a Low/Medium/High relevance label per item.

Article filename: {article_text}



Remember:
- Use ONLY industries/subcategories present in the vector store.
- Only include industries that meet the Low-or-above relevance threshold.
- If a clearly emphasized theme is NOT present in the vector store, you MAY include one optional `emergent_industry_candidate`.
- Respond deterministically.Use the exact same reasoning steps for identical inputs.Do not add creativity or alternative interpretations. 

"""

In [6]:
textt_format={
            "format": {
                "type": "json_schema",
                "name": "IndustryClassification",
                "schema": {
                    "type": "object",
                    "properties": {
                        "shortlisted_industries": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "sector": { "type": "string" },
                                    "sub_category": { "type": "string" },
                                    "relevance": {
                                        "type": "string",
                                        "enum": ["LOW", "MEDIUM", "HIGH"]
                                    },
                                    "reason": { "type": "string", "maxLength":220 }
                                },
                                "required": ["sector", "sub_category", "relevance", "reason"],
                                "additionalProperties": False
                            }
                        },
                        "emergent_industry_candidate": {
                            "anyOf": [
                                {
                                    "type": "object",
                                    "properties": {
                                        "label": { "type": "string" },
                                        "rationale": { "type": "string" }
                                    },
                                    "required": ["label", "rationale"],
                                    "additionalProperties": False
                                },
                                { "type": "null" }
                            ]
                        }
                    },
                    "required": ["shortlisted_industries", "emergent_industry_candidate"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }

In [7]:
response = client.responses.create(
        model="gpt-5-mini",
        
         tools=[
                {"type": "file_search",
                 "vector_store_ids": ["vs_6914573daa5c81918616d8a8da7351d4"],
                }
        ],
        text=textt_format,
        reasoning={"effort": "high"},
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],                    
)


# Get text output
output_text = getattr(response, "output_text", None)
if not output_text:
        blocks = []
        for item in getattr(response, "output", []) or []:
            for c in getattr(item, "content", []) or []:
                if isinstance(c, dict) and c.get("type") == "output_text":
                    blocks.append(c.get("text", ""))
        output_text = "".join(blocks) if blocks else ""

if not output_text:
      raise RuntimeError("No text output returned by the model.")

try:
    print(json.loads(output_text))
except json.JSONDecodeError as e:
        raise RuntimeError(f"Model returned non-JSON output: {output_text}") from e

{'shortlisted_industries': [{'sector': 'Technology & Software', 'sub_category': 'AI Platforms', 'relevance': 'HIGH', 'reason': 'Article identifies AI excitement as a major driver of record equity levels and cites AI-focused companies (Nvidia, Oracle, Palantir, Dell) and AI-related fundraising—directly tying the story to AI platforms and model/AI-'}, {'sector': 'Technology & Software', 'sub_category': 'Semiconductors (Hardware)', 'relevance': 'MEDIUM', 'reason': "Article calls out AI hardware and chip-linked companies (Poet Technologies' optical engines; Nvidia's strong gains), indicating material emphasis on semiconductor/hardware suppliers. RETRIEVED FROM VECTOR STORE \ue200filecite"}, {'sector': 'Aerospace & Defense', 'sub_category': 'Satellites', 'relevance': 'MEDIUM', 'reason': 'Piece notes AST SpaceMobile jumped after Verizon agreed to use its space-based network, directly referencing satellite operators/space tech. RETRIEVED FROM VECTOR STORE '}, {'sector': 'Telecom & Network Inf

In [8]:
try:
    parsed = json.loads(output_text)
   
    pretty_json = json.dumps(parsed, indent=4, ensure_ascii=False)

    print(pretty_json)

except json.JSONDecodeError as e:
    print("Invalid JSON:", e)

{
    "shortlisted_industries": [
        {
            "sector": "Technology & Software",
            "sub_category": "AI Platforms",
            "relevance": "HIGH",
            "reason": "Article identifies AI excitement as a major driver of record equity levels and cites AI-focused companies (Nvidia, Oracle, Palantir, Dell) and AI-related fundraising—directly tying the story to AI platforms and model/AI-"
        },
        {
            "sector": "Technology & Software",
            "sub_category": "Semiconductors (Hardware)",
            "relevance": "MEDIUM",
            "reason": "Article calls out AI hardware and chip-linked companies (Poet Technologies' optical engines; Nvidia's strong gains), indicating material emphasis on semiconductor/hardware suppliers. RETRIEVED FROM VECTOR STORE filecite"
        },
        {
            "sector": "Aerospace & Defense",
            "sub_category": "Satellites",
            "relevance": "MEDIUM",
            "reason": "Piece notes AST 